# Solutions — TPC-H Lineitem Analysis (Medium 04)

**Dataset:** `samples.tpch.lineitem`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

lineitem = spark.table("samples.tpch.lineitem")

## Solution 1 — Net Revenue per Line Item

In [ ]:
# Why: net_revenue = price * (1 - discount) * (1 + tax)
result_1 = (
    lineitem
    .withColumn(
        "net_revenue",
        F.round(F.col("l_extendedprice") * (1 - F.col("l_discount")) * (1 + F.col("l_tax")), 2)
    )
    .select("l_orderkey","l_linenumber","l_extendedprice","l_discount","l_tax","net_revenue")
)

## Solution 2 — Revenue & Avg Discount per Ship Mode

In [ ]:
result_2 = (
    lineitem
    .groupBy("l_shipmode")
    .agg(
        F.round(F.sum("l_extendedprice"), 2).alias("total_revenue"),
        F.round(F.avg("l_discount"), 4).alias("avg_discount")
    )
    .orderBy(F.col("total_revenue").desc())
)

## Solution 3 — Line Count, Qty, Revenue by Return Flag & Line Status

In [ ]:
result_3 = (
    lineitem
    .groupBy("l_returnflag","l_linestatus")
    .agg(
        F.count("*").alias("line_count"),
        F.round(F.sum("l_quantity"), 0).alias("total_qty"),
        F.round(F.sum("l_extendedprice"), 2).alias("total_revenue")
    )
    .orderBy("l_returnflag","l_linestatus")
)

## Solution 4 — Revenue per Ship Month

In [ ]:
result_4 = (
    lineitem
    .withColumn("ship_year", F.year("l_shipdate"))
    .withColumn("ship_month", F.month("l_shipdate"))
    .groupBy("ship_year","ship_month")
    .agg(
        F.round(F.sum("l_extendedprice"), 2).alias("total_revenue"),
        F.round(F.sum("l_quantity"), 0).alias("total_quantity"),
        F.round(F.avg("l_discount"), 4).alias("avg_discount")
    )
    .orderBy("ship_year","ship_month")
)

## Solution 5 — High Discount High Quantity Items

In [ ]:
# Why: both conditions must hold — use & (and) not 'and' keyword
result_5 = (
    lineitem
    .filter((F.col("l_discount") > 0.05) & (F.col("l_quantity") > 20))
    .select("l_orderkey","l_partkey","l_quantity","l_discount","l_extendedprice")
)

## Solution 6 — Cumulative Revenue by Ship Mode Ordered by Ship Date

In [ ]:
# Why: partitionBy shipmode so cumulative sum resets per mode
w = Window.partitionBy("l_shipmode").orderBy("l_shipdate").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result_6 = (
    lineitem
    .select(
        "l_shipdate","l_shipmode","l_extendedprice",
        F.round(F.sum("l_extendedprice").over(w), 2).alias("cumulative_revenue")
    )
    .orderBy("l_shipmode","l_shipdate")
)

## Solution 7 — Orders with More Line Items Than Average

In [ ]:
# Why: compute global avg line items, then filter orders above it
avg_items = lineitem.groupBy("l_orderkey").agg(F.count("*").alias("line_item_count"))
global_avg = avg_items.agg(F.avg("line_item_count")).collect()[0][0]
result_7 = (
    avg_items
    .withColumn("avg_line_items", F.lit(round(global_avg, 2)))
    .filter(F.col("line_item_count") > F.col("avg_line_items"))
    .orderBy(F.col("line_item_count").desc())
)